# Multi level Perceptron (MLP): Using Numpy and Pandas

* Develop and train a MLP model to accurately classify MNIST data. 
* Use one hidden layer, one input layer and one output layer. 
* Test and choose an appropriate activation function, Optimizer and loss function. 
* Implement forward-propagation and Back-propagation.

In [20]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

## Loading MNIST data
Load data and mention data properties

In [21]:
# Loading test and train data
train_data_input = pd.read_csv('MNIST_CSV/mnist_train.csv', header=None)
test_data_input = pd.read_csv('MNIST_CSV/mnist_test.csv', header=None)

In [22]:
# Data properties

print('Data shape : ', train_data_input.shape)

print('Image length : ', train_data_input.iloc[1,1:].shape)

print('Feature Data type : ', type(train_data_input.iloc[1,1]))

print('Class Data type : ', type(train_data_input.iloc[0,0]))


Data shape :  (60000, 785)
Image length :  (784,)
Feature Data type :  <class 'numpy.int64'>
Class Data type :  <class 'numpy.int64'>


In [23]:
# Separating Class and Normalizing data from -0.5 to 0.5

train_data = (train_data_input.iloc[:,1:] / 255).to_numpy() - 0.5

y_train = train_data_input.iloc[:,0].to_numpy()

test_data = (test_data_input.iloc[:,1:]  / 255).to_numpy() - 0.5

y_test = test_data_input.iloc[:,0].to_numpy()

print(f'Minimum and Maximum feature values: {train_data.min()}, {train_data.max()}')



Minimum and Maximum feature values: -0.5, 0.5


## Building the FC1 layer

* Building the first fully connected layer of MPL. Using Weight and bias matrices to expand
* Initializing weight with random values to break neuron symmetry



In [24]:
# Initializing weight 
input_weights = np.random.randn(784, 256) * np.sqrt(2/784)

# Initializing bias  
input_bias = np.ones((1, 256)) * 0.01

# Input layer function
def forward_fc1(input_array = None):

    output_matrix = np.matmul(input_array , input_weights) + input_bias

    return output_matrix

FC1_output = forward_fc1 (train_data)


# Adding activation layer

Introducing non linearity in neurons through hidden layer to make model functionally strong. Using ReLU, Sigmoid and Tanh as activation functions. Compressing the layer to 512 neurons.

In [25]:
# Creating a function for activation layer

# Activation layer function 
def Layer_activation(input_matrix = None, activation = 'ReLU'):

    if activation == 'ReLU':

        output_matrix = np.maximum(0, input_matrix)
    
        return output_matrix
    
    if activation == 'Leaky ReLU':

        output_matrix = np.maximum(0.01 * input_matrix, input_matrix)
    
        return output_matrix

Activation_output =  Layer_activation (FC1_output)


# Building FC2 layer

* Building a second hidden fully connected layer to ensure gradual compression of neurons.
* Initializing weight with random values to break neuron symmetry


In [26]:
# Layer_2 is a fully connected perceptron layer consisting of 256 neurons

# Initializing weight 
FC2_weights = np.random.randn   (256,128) * np.sqrt(2/256)

#Initializing bias 
FC2_Bias = np.ones((1,128)) *.01

# Fc2 layer function
def forward_fc2 (input_matrix = None):
    
    output_matrix = np.matmul( input_matrix , FC2_weights) + FC2_Bias

    return output_matrix

FC2_output = forward_fc2 (Activation_output)



# Activation layer

Introducing another non linearity in neurons through hidden layer to make model functionally strong. Using ReLU, Sigmoid and Tanh as activation functions. 

Z3 = ReLU( ReLU( XW1 ) W2 ) W3

In [27]:
# Creating a function for activation layer

Activation_output_2 =  Layer_activation (FC2_output)

# Output Layer

Output layer compress the FC2 output further into 10 classes (digits)

In [28]:
# Initializing weight and bias matrices with random values to break neuron symmetry

output_weights = np.random.randn(128,10) * np.sqrt(2/128)

output_bias = np.ones((1,10)) * 0.01

# Output layer function 
def forward_output (input_matrix = None, activation = 'softmax'):

    output_matrix = np.matmul( input_matrix , output_weights ) + output_bias

    if activation == 'sigmoid': 

        output_matrix = 1 / ( 1 + np.exp(-output_matrix) )

    if activation == 'softmax':

        z = output_matrix - output_matrix.max( keepdims = True)

        exp_z = np.exp(output_matrix)

        output_matrix = exp_z / exp_z.sum( axis=1, keepdims = True)

    return output_matrix

predictions = forward_output (Activation_output_2)


# One hot encoding for target variables for all training data
Converting the target array into 2d one hot encoded array for easier loss calculation 

In [29]:
One_hot_encoded_train = np.zeros((60000,10))

One_hot_encoded_train[np.arange(60000),y_train] =1

print(One_hot_encoded_train)

[[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]]


# Variance analysis

Variance should neither shrink or increase by order of a magnitude through each layer.

In [30]:
# Comparing variance

print('Input variance: ', np.mean(np.var(train_data, axis=0)))
print('FC1 Layer variance: ', np.mean(np.var(FC1_output, axis=0)))
print('Activation Layer 1 variance: ', np.mean(np.var(Activation_output, axis=0)))
print('FC2 Layer variance: ', np.mean(np.var(FC2_output, axis=0)))
print('Activation Layer variance: ', np.mean(np.var(Activation_output_2, axis=0)))      



Input variance:  0.06725132078460057
FC1 Layer variance:  0.1368450903108544
Activation Layer 1 variance:  0.05671148645772975
FC2 Layer variance:  0.11245330511450301
Activation Layer variance:  0.04737427701783054


## Conclusions of variance analysis

* ReLU activation layer only activated when input range was changed from (0,1) to (0.5,0.5)
* ReLU Reduces variance to 0 when bias is zero. Bias needs to be small positive value
* Leaky ReLU performs better is preventing the variance from turning to 0
* Zero centered input and initial weights are essential for variance to not collapse in the activation (ReLU) layer

# Loss function
Using cross entropy as loss function for simple loss gradient function.
* Loss in cross entropy is always positive number as predictions range from (0, 1)

In [31]:
# Implementing cross entropy loss function

def cross_entropy (input_array = None , target = None):

    loss = target * np.log(input_array) * -1
    
    return loss

print(cross_entropy(predictions, One_hot_encoded_train))


[[0.         0.         0.         ... 0.         0.         0.        ]
 [2.55181881 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         2.03557294 0.        ]]


# Gradient calculations




## Backpropagation — Gradients of Weights and Biases w.r.t Loss

Gradient is defined as the **rate of change of loss with respect to a variable**. Using the chain rule, gradients are propagated in reverse through the network: **Output Layer → FC2 → FC1**.

**Notation:**
- `z` = pre-activation output
- `a` = post-activation output (after ReLU)
- `dz` = dL/dz &nbsp;|&nbsp; `dw` = dL/dW &nbsp;|&nbsp; `db` = dL/db &nbsp;|&nbsp; `da` = dL/da

In [ ]:


# --------------------------------------------------------------
# OUTPUT LAYER (FC3)
# Loss: cross-entropy | Activation: softmax
# dL/dz3 = predictions - y  (softmax + cross-entropy combined)
# --------------------------------------------------------------

# Gradient of loss w.r.t. output layer pre-activation
dz3 = predictions - One_hot_encoded_train

# Gradient of loss w.r.t. output layer weights
# dL/dW3 = A2^T @ dz3
dw3 = np.transpose(Activation_output_2) @ dz3

# Gradient of loss w.r.t. output layer bias
db3 = sum(dz3)


# --------------------------------------------------------------
# SECOND HIDDEN LAYER (FC2)
# Activation: ReLU  →  ReLU gradient = 1 if z > 0, else 0
# --------------------------------------------------------------

# Gradient of loss w.r.t. FC2 post-activation (backdrop through output weights)
# dL/dA2 = dz3 @ W3^T
da2 = dz3 @ np.transpose(output_weights)

# Gradient of loss w.r.t. FC2 pre-activation (backprop through ReLU)
# ReLU derivative: pass gradient only where FC2 output was positive
dz2 = da2 * (FC2_output > 0)

# Gradient of loss w.r.t. FC2 weights
# dL/dW2 = A1^T @ dz2
dw2 = np.transpose(Activation_output) @ dz2

# Gradient of loss w.r.t. FC2 bias
db2 = sum(dz2)


# --------------------------------------------------------------
# FIRST HIDDEN LAYER (FC1)
# Activation: ReLU  →  ReLU gradient = 1 if z > 0, else 0
# --------------------------------------------------------------

# Gradient of loss w.r.t. FC1 post-activation (backprop through FC2 weights)
# dL/dA1 = dz2 @ W2^T
da1 = dz2 @ np.transpose(FC2_weights)

# Gradient of loss w.r.t. FC1 pre-activation (backprop through ReLU)
# ReLU derivative: pass gradient only where FC1 output was positive
dz1 = da1 * (FC1_output > 0)

# Gradient of loss w.r.t. FC1 weights
# dL/dW1 = X^T @ dz1
dw1 = np.transpose(train_data) @ dz1

# Gradient of loss w.r.t. FC1 bias
db1 = sum(dz1)

In [37]:
print(np.sum(np.ones((5,3)),keepdims=True))

[[15.]]
